## Reproducibility & AI-assistance disclosure

Part of **`dislocation-speech-translation-fr-en`**, companion to the M2 thesis *Dislocation under Translation: A Corpus Study of Whisper and XLS-R on Spontaneous Spoken French* (Zoé de Vries, Université Paris Cité, 2026; supervised by Pr. Ballier).

**AI-assistance disclosure.** The explanatory **comments and markdown** were drafted with the help of a large language model (LLM) and reviewed by the author.

**Pipeline order:** `01_whisper` → `02_xlsr` → `03_align` → `04_score`.


# Whisper — décodage long-form continu (audio complet)

Pipeline réécrit pour le décodage continu réel, plus de segments pré-découpés.

1. Téléchargement de l'enregistrement source CFPP2000 (identique au notebook XLS-R → audio reproductible).
2. Passe `transcribe` sur l'audio entier → segments alignés sur les pauses. **Cette segmentation est la grille canonique** de toute la thèse.
3. Passe `translate` sur l'audio entier → SRT de traduction continue (sortie demandée par Ballier).
4. Pour chaque segment canonique : on découpe l'audio à ses bornes et on fait tourner Whisper `translate` sur la tranche → traduction EN par segment, comparable à XLS-R.

**Contrat inter-notebooks.** Ce notebook écrit `whisper_grid_manifest.csv` :
`segment_id, t_start, t_end, dur_s, whisper_fr, whisper_en`.
Le notebook XLS-R **lit ce fichier** (à déposer dans le Drive) et applique exactement les mêmes bornes.

> Compromis assumé : la traduction EN par segment est produite sur une tranche isolée (moins de contexte que la passe continue). C'est le prix de la comparabilité segment-à-segment entre modèles. La passe `translate` continue est conservée à part (`*_translate.srt`) comme traduction de référence à contexte plein.


In [ ]:
# Environment bootstrap: make user site-packages importable; log interpreter, whisper path, CUDA.

# ── 0. Environnement (bootstrap user-site, inchangé) ───────────────────────────
import sys, site, os, pathlib, inspect, json, time, traceback
from pathlib import Path
import whisper, torch
from tqdm import tqdm

USER_SITE = site.getusersitepackages()
if os.path.isdir(USER_SITE) and USER_SITE not in sys.path:
    sys.path.insert(0, USER_SITE)

print("Python   :", sys.executable)
print("whisper  :", os.path.dirname(inspect.getfile(whisper)))
print("CUDA     :", torch.cuda.is_available())


In [ ]:
# Config. SOURCE_URL = public CFPP2000 recording (downloaded at run time, never redistributed).
# MODEL_NAME='medium' = the thesis variant. The transcribe pass defines the 874-segment canonical grid.

# ── 1. CONFIG ──────────────────────────────────────────────────────────────────
# Source identique au notebook XLS-R (même URL) → audio strictement reproductible.
SOURCE_URL = "http://cfpp2000.univ-paris3.fr/data/public/11eme/Anita_MUSSO_F_46_11e/Anita_MUSSO_F_46_11e.wav"
RAW_PATH   = Path("data/Anita_MUSSO_F_46_11e_raw.wav")
OUT_DIR    = Path("data/whisper_output")

# Modèle Whisper — la qualité de segmentation pilote désormais TOUT le pipeline.
MODEL_NAME = "medium"        # version actuelle — on tente d'abord celle-ci
# MODEL_NAME = "large-v3"    # + lent / + de VRAM, segmentation et timestamps nettement plus fiables

LANG   = "fr"
SR     = 16_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = OUT_DIR / "whisper_grid_manifest.csv"
print("Modèle :", MODEL_NAME, "| Device :", DEVICE)
print("Manifest (à copier ensuite dans le Drive pour XLS-R) :", MANIFEST_PATH)


In [ ]:
# Download the source WAV once (idempotent) and load it 16 kHz mono for per-segment slicing.

# ── 2. Téléchargement de la source ─────────────────────────────────────────────
import requests, librosa, numpy as np

if not RAW_PATH.exists():
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("Téléchargement de l'enregistrement source…")
    with requests.get(SOURCE_URL, stream=True, timeout=180) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(RAW_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                f.write(chunk); done += len(chunk)
                if total:
                    print(f"  {done/1024**2:.1f} / {total/1024**2:.1f} MB", end="\r")
    print("\nEnregistré :", RAW_PATH)
else:
    print("Source déjà présente :", RAW_PATH)

# Chargement unique en mémoire (réutilisé pour le découpage par segment)
print("Chargement 16 kHz mono…")
audio_full, _ = librosa.load(RAW_PATH, sr=SR, mono=True)
print(f"  Durée totale : {len(audio_full)/SR:.1f} s")


In [ ]:
# Load the Whisper model and a minimal SRT writer.

# ── 3. Modèle + utilitaire SRT ─────────────────────────────────────────────────
model = whisper.load_model(MODEL_NAME, device=DEVICE)

def save_srt(segments, out_path):
    def fmt(ts):
        h = int(ts // 3600); m = int((ts % 3600) // 60)
        s = int(ts % 60);    ms = int(round((ts - int(ts)) * 1000))
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    lines = []
    for i, seg in enumerate(segments, 1):
        lines += [str(i),
                  f"{fmt(seg.get('start', 0))} --> {fmt(seg.get('end', 0))}",
                  seg.get("text", "").strip(), ""]
    Path(out_path).write_text("\n".join(lines), encoding="utf-8")
    print("SRT écrit :", out_path)


In [ ]:
# Pass 1 - TRANSCRIBE (whole recording, word_timestamps=True). Pause-aligned boundaries become the
# canonical grid; segments <= 0.05 s dropped. Per-segment EN is stored as whisper_en_per_seg.

# ── 4. Passe TRANSCRIBE → grille canonique + SRT ───────────────────────────────
# word_timestamps=True : bornes de segments plus fiables (la grille est critique).
print("Passe transcribe (audio entier)…")
res_tr = model.transcribe(audio_full, language=LANG, task="transcribe",
                          word_timestamps=True, verbose=False)

segments = res_tr["segments"]
save_srt(segments, OUT_DIR / "Anita_MUSSO_transcribe.srt")

# Construction de la grille canonique
import csv
grid = []
for i, seg in enumerate(segments, 1):
    start, end = float(seg["start"]), float(seg["end"])
    if end - start <= 0.05:          # garde-fou : segment vide / dégénéré
        continue
    grid.append({
        "segment_id": f"seg_{i:04d}",
        "t_start": round(start, 3),
        "t_end":   round(end, 3),
        "dur_s":   round(end - start, 3),
        "whisper_fr": " ".join(seg.get("text", "").split()),
        "whisper_en_per_seg": "",
    })

print(f"{len(grid)} segments canoniques.")
durs = [g["dur_s"] for g in grid]
print(f"Durées — min {min(durs):.2f}s | max {max(durs):.2f}s | moy {sum(durs)/len(durs):.2f}s")
print(f"Segments > 28s (seront sous-découpés côté XLS-R) : {sum(d > 28 for d in durs)}")


In [ ]:
# Pass 2 - TRANSLATE continuous (full context). Archived as SRT; re-indexed onto the grid later.

# ── 5. Passe TRANSLATE continue → SRT de référence à contexte plein ────────────
print("Passe translate (audio entier)…")
res_en = model.transcribe(audio_full, language=LANG, task="translate",
                          word_timestamps=True, verbose=False)
save_srt(res_en["segments"], OUT_DIR / "Anita_MUSSO_translate.srt")


In [ ]:
# Pass 3 - per-segment TRANSLATE: each slice translated in isolation -> whisper_en_per_seg,
# segment-for-segment comparable with XLS-R; slices < 0.2 s left empty.

# ── 6. Traduction EN par segment canonique (tranche audio découpée) ────────────
# Whisper translate sur chaque tranche, pour rester comparable à XLS-R (même grille).
for g in tqdm(grid, desc="Whisper EN / segment"):
    a = int(g["t_start"] * SR); b = int(g["t_end"] * SR)
    clip = audio_full[a:b]
    if len(clip) < int(0.2 * SR):          # tranche trop courte → vide
        g["whisper_en_per_seg"] = ""
        continue
    try:
        out = model.transcribe(clip, language=LANG, task="translate", verbose=None)
        g["whisper_en_per_seg"] = " ".join((out.get("text", "") or "").split())
    except Exception as e:
        print("  ✗", g["segment_id"], e)
        g["whisper_en_per_seg"] = ""


In [ ]:
# Write the shared manifest (segment_id, t_start, t_end, dur_s, whisper_fr, whisper_en_per_seg).
# MANUAL STEP: copy it to the location read by notebook 02.

# ── 7. Écriture du manifest partagé ────────────────────────────────────────────
cols = ["segment_id", "t_start", "t_end", "dur_s", "whisper_fr", "whisper_en_per_seg"]
with open(MANIFEST_PATH, "w", encoding="utf-8", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=cols)
    wr.writeheader()
    for g in grid:
        wr.writerow(g)

print("Manifest écrit :", MANIFEST_PATH, f"({len(grid)} lignes)")
print("\n>>> ÉTAPE MANUELLE : copier ce fichier dans le Google Drive,")
print(">>> puis renseigner MANIFEST_PATH dans le notebook XLS-R.")
for g in grid[:5]:
    print()
    print(g["segment_id"], f"[{g['t_start']}–{g['t_end']}]")
    print("  FR:", g["whisper_fr"][:120])
    print("  EN:", g["whisper_en_per_seg"][:120])


---
## Enrichissement — `whisper_en_cont` (autonome, sans ré-inférence)

À exécuter **après** le pipeline, ou seule sur un kernel neuf : lit `MANIFEST_PATH`
(cellule CONFIG) + `Anita_MUSSO_translate.srt` déjà sur disque, ajoute la colonne
`whisper_en_cont` (passe translate continue re-indexée sur la grille par recouvrement
temporel) et réécrit le manifest. Idempotent.

In [ ]:
# Enrichment (idempotent): re-index the continuous TRANSLATE SRT onto the grid -> whisper_en_cont.

import re as _re, pandas as pd
from pathlib import Path

SRT_PATH = OUT_DIR / "Anita_MUSSO_translate.srt"

def parse_srt(p):
    cues=[]
    for b in Path(p).read_text(encoding="utf-8").strip().split("\n\n"):
        ls=[l for l in b.splitlines() if l.strip()]
        if len(ls)<2: continue
        m=_re.match(r"(\d\d):(\d\d):(\d\d),(\d\d\d)\s*-->\s*(\d\d):(\d\d):(\d\d),(\d\d\d)", ls[1])
        if not m: continue
        g=list(map(int,m.groups()))
        s=g[0]*3600+g[1]*60+g[2]+g[3]/1000
        e=g[4]*3600+g[5]*60+g[6]+g[7]/1000
        cues.append((s,e," ".join(" ".join(ls[2:]).split())))
    return cues

cues=parse_srt(SRT_PATH)
df=pd.read_csv(MANIFEST_PATH)

def map_cont(s,e):
    hit=[(cs,t) for cs,ce,t in cues if t and min(e,ce)-max(s,cs)>0]
    hit.sort()
    return " ".join(t for _,t in hit)

df["whisper_en_cont"]=[map_cont(r.t_start,r.t_end) for r in df.itertuples()]
df.to_csv(MANIFEST_PATH,index=False,encoding="utf-8")

n=len(df)
print(f"{len(cues)} cues | {n} segments")
print("whisper_en_per_seg vides :", int(df.whisper_en_per_seg.fillna('').str.strip().eq('').sum()))
print("whisper_en_cont vides :", int(df.whisper_en_cont.fillna('').str.strip().eq('').sum()))
print("Manifest réécrit :", MANIFEST_PATH)
